In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/chiragmc/data-splits/train_ids.csv
/kaggle/input/datasets/chiragmc/data-splits/val_ids.csv
/kaggle/input/datasets/chiragmc/data-splits/test_ids.csv
/kaggle/input/datasets/chiragmc/python-files/preprocessing.py
/kaggle/input/datasets/chiragmc/python-files/metrics.py
/kaggle/input/datasets/chiragmc/python-files/dataset.py
/kaggle/input/datasets/chiragmc/python-files/model_architecture.py
/kaggle/input/datasets/chiragmc/plasticc-unblinded-data/plasticc_test_set_batch10.csv
/kaggle/input/datasets/chiragmc/plasticc-unblinded-data/plasticc_test_set_batch4.csv
/kaggle/input/datasets/chiragmc/plasticc-unblinded-data/plasticc_test_set_batch7.csv
/kaggle/input/datasets/chiragmc/plasticc-unblinded-data/plasticc_test_set_batch5.csv
/kaggle/input/datasets/chiragmc/plasticc-unblinded-data/plasticc_test_set_batch9.csv
/kaggle/input/datasets/chiragmc/plasticc-unblinded-data/plasticc_test_set_batch1.csv
/kaggle/input/datasets/chiragmc/plasticc-unblinded-data/plasticc_test_set_bat

# TFT Training Pipeline: PLAsTiCC Unblinded Data

This notebook orchestrates the training of a Temporal Fusion Transformer (TFT) on the  unblinded PLAsTiCC astronomical time-series dataset. 

## 1. Project Recap: What Happened Before This Notebook?
Before reaching the modeling phase, we established a standardized data pipeline:
* **EDA (`00_data_exploration_EDA.ipynb`):** We analyzed the 3.5M object universe and identified extreme class imbalance and severe temporal intermittency (gaps between telescope detections).
* **Data Stratification (`01_stratified_data_split.ipynb`):** To handle the 22 GB dataset size, we performed a stratified split based on the true target classes. We exported lightweight ID ledgers (`train_ids.csv`, `val_ids.csv`, `test_ids.csv`) which act as our absolute "Source of Truth".

## 2. Our Custom PyTorch Toolkit
To keep this notebook clean, all complex mathematical transformations and PyTorch class definitions have been isolated into four modular `.py` files:
* **`metrics.py`:** Contains our evaluation suite, including the official PLAsTiCC Weighted Log-Loss (which penalizes rare anomalies like Kilonovae twice as heavily) and Macro F1.
* **`preprocessing.py`:** Handles memory-safe chunked loading, applies Robust Scaling to the flux anomalies, and engineers the continuous time gap ($\Delta t$) feature.
* **`dataset.py`:** Packages the tabular data into 3D GPU matrices, padding all sequences to 350 steps and generating boolean attention masks so the model ignores the artificial padding.
* **`model_architecture.py`:** Defines the modified TFT topology, replacing the standard continuous forecasting head with a Multi-Layer Perceptron (MLP) optimized for sequence classification.

## Execution Plan
* **Phase 1: Environment & Module Mapping:** Establish paths to the 22GB dataset and our custom `.py` architecture scripts.
* **Phase 2: Unit Testing:** Run a lightweight 10-object dummy batch through the custom `dataset` and `model_architecture` modules to verify tensor shapes and memory allocation.
* **Phase 3: Massive Data Instantiation:** Load the stratified `train_ids` and instantiate the PyTorch `WeightedRandomSampler` to force physical class diversity in every batch.
* **Phase 4: Model Initialization:** Initialize the modified TFT classification head, AdamW optimizer, and custom Weighted Log-Loss metric.
* **Phase 5: The Training Loop:** Execute the FP16 Mixed Precision training loop with Early Stopping based on the validation Log-Loss.
* **Phase 6: Artifact Export:** Generate test set predictions and save the `.pth` weights for deployment.

## Phase 1: Environment Setup and Module Import
Map the exact Kaggle input directories for the raw data, the data splits, and the custom Python files. Append the scripts directory to the system path to enable native imports.

In [2]:
import sys
import os
import torch
import pandas as pd

# 1. Define Kaggle Input Paths
DATA_DIR = '/kaggle/input/datasets/chiragmc/plasticc-unblinded-data'
SCRIPTS_DIR = '/kaggle/input/datasets/chiragmc/python-files'
SPLITS_DIR = '/kaggle/input/datasets/chiragmc/data-splits'

print("Checking data directories...")
try:
    print(f"Data files found: {os.listdir(DATA_DIR)[:3]}...")
    print(f"Splits found: {os.listdir(SPLITS_DIR)}")
    print(f"Scripts found: {os.listdir(SCRIPTS_DIR)}")
except FileNotFoundError as e:
    print(f"Path Error: {e}\nCheck if the Kaggle dataset is attached to the notebook.")

# 2. Append Custom Scripts to System Path
if SCRIPTS_DIR not in sys.path:
    sys.path.append(SCRIPTS_DIR)

print("\nImporting custom modules...")
try:
    import preprocessing
    import dataset
    import model_architecture
    import metrics
    print("Success: All custom .py modules imported perfectly.")
except ImportError as e:
    print(f"Import Error: {e}")
    print("Ensure the Python_files dataset is structured correctly without subfolders.")

# 3. Verify Compute
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nCompute Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Checking data directories...
Data files found: ['plasticc_test_set_batch10.csv', 'plasticc_test_set_batch4.csv', 'plasticc_test_set_batch7.csv']...
Splits found: ['train_ids.csv', 'val_ids.csv', 'test_ids.csv']
Scripts found: ['preprocessing.py', 'metrics.py', 'dataset.py', 'model_architecture.py']

Importing custom modules...
Success: All custom .py modules imported perfectly.

Compute Device: cuda
GPU Count: 2
GPU Name: Tesla T4


## Phase 2: Pipeline Unit Testing
Before loading the massive 100,000-object training set into RAM, we will isolate just **5 objects** from the `train_ids.csv`. We will pass these 5 objects through our `preprocessing.py` logic, feed them into the `dataset.py` DataLoader to verify padding/masking, and pass the resulting tensors through a dummy forward pass of `model_architecture.py`. 

If this block runs without memory allocation or tensor shape errors, the mathematical pipeline is perfectly sound.

In [3]:
import time
import pandas as pd
import torch
from torch.utils.data import DataLoader

# Explicitly import modules to prevent NameError
import preprocessing
from dataset import PLAsTiCCTFTDataset
from model_architecture import TFTClassifier

print("--- STARTING PHASE 2 UNIT TESTS ---\n")

# 1. Isolate 5 IDs GUARANTEED to be in the train_lightcurves file
meta_path = os.path.join(DATA_DIR, 'plasticc_train_metadata.csv')
lc_path = os.path.join(DATA_DIR, 'plasticc_train_lightcurves.csv')

# Read exactly 5 IDs from the actual train metadata
dummy_meta_df = pd.read_csv(meta_path, nrows=5)
dummy_ids = dummy_meta_df['object_id'].tolist()
print(f"Extracted 5 dummy IDs guaranteed to exist: {dummy_ids}")

# Create a temporary dummy CSV to act as our 'valid_ids_path' for the preprocessing function
dummy_ids_path = '/kaggle/working/dummy_ids.csv'
pd.DataFrame({'object_id': dummy_ids}).to_csv(dummy_ids_path, index=False)

# 2. Test Preprocessing
print("\nTesting preprocessing.py (Reading lightcurves and computing delta_t)...")
start_time = time.time()

processed_dummy_df = preprocessing.build_processed_dataset(lc_path, meta_path, dummy_ids_path)

print(f"Preprocessing successful! Generated DataFrame shape: {processed_dummy_df.shape}")
print(f"Time taken: {time.time() - start_time:.2f} seconds")

# 3. Test PyTorch Dataset & DataLoader
print("\nTesting dataset.py (Padding, Masking, and Tensor generation)...")
dummy_dataset = PLAsTiCCTFTDataset(processed_dummy_df, max_seq_len=350)
dummy_loader = DataLoader(dummy_dataset, batch_size=2, shuffle=False)

# Grab one batch
batch = next(iter(dummy_loader))
print("Batch shapes:")
print(f" - static:   {batch['static'].shape}   -> Expected: [2, 3]")
print(f" - dyn_cont: {batch['dyn_cont'].shape} -> Expected: [2, 350, 4]")
print(f" - dyn_cat:  {batch['dyn_cat'].shape}  -> Expected: [2, 350]")
print(f" - mask:     {batch['mask'].shape}     -> Expected: [2, 350]")
print(f" - target:   {batch['target'].shape}   -> Expected: [2]")

# 4. Test Model Forward Pass
print("\nTesting model_architecture.py (TFT Forward Pass)...")
# Send model to the device assigned in Phase 1
model = TFTClassifier(num_classes=dummy_dataset.num_classes).to(device)

static_tensor = batch['static'].to(device)
dyn_cont_tensor = batch['dyn_cont'].to(device)
dyn_cat_tensor = batch['dyn_cat'].to(device)
mask_tensor = batch['mask'].to(device)

with torch.no_grad():
    logits = model(static_tensor, dyn_cont_tensor, dyn_cat_tensor, mask_tensor)

print(f"Forward pass successful! Output logits shape: {logits.shape}")
print("\n--- ALL PHASE 2 TESTS PASSED ---")

--- STARTING PHASE 2 UNIT TESTS ---

Extracted 5 dummy IDs guaranteed to exist: [615, 713, 730, 745, 1124]

Testing preprocessing.py (Reading lightcurves and computing delta_t)...
Preprocessing successful! Generated DataFrame shape: (1735, 33)
Time taken: 1.07 seconds

Testing dataset.py (Padding, Masking, and Tensor generation)...
Batch shapes:
 - static:   torch.Size([2, 3])   -> Expected: [2, 3]
 - dyn_cont: torch.Size([2, 350, 4]) -> Expected: [2, 350, 4]
 - dyn_cat:  torch.Size([2, 350])  -> Expected: [2, 350]
 - mask:     torch.Size([2, 350])     -> Expected: [2, 350]
 - target:   torch.Size([2])   -> Expected: [2]

Testing model_architecture.py (TFT Forward Pass)...
Forward pass successful! Output logits shape: torch.Size([2, 4])

--- ALL PHASE 2 TESTS PASSED ---


## Phase 3: Massive Data Instantiation & Balancing
Because our stratified IDs are scattered across the entire unblinded universe, we must iterate through all 12 lightcurve files and both metadata files to assemble our training and validation sets. 

Once loaded, we will instantiate a PyTorch `WeightedRandomSampler`. Because Type Ia Supernovae dominate the universe, this sampler forces every single batch to contain a physically diverse representation of rare anomalies like Kilonovae.

In [4]:
import glob
from torch.utils.data import WeightedRandomSampler

print("--- STARTING PHASE 3: DATA PIPELINE ---")

# 1. Load the Target IDs
train_ids_df = pd.read_csv(os.path.join(SPLITS_DIR, 'train_ids.csv'))
val_ids_df = pd.read_csv(os.path.join(SPLITS_DIR, 'val_ids.csv'))
all_valid_ids = pd.concat([train_ids_df, val_ids_df])

# 2. Extract Static Metadata from all files
print("Scanning Metadata files...")
meta_files = [os.path.join(DATA_DIR, 'plasticc_train_metadata.csv'), 
              os.path.join(DATA_DIR, 'plasticc_test_metadata.csv')]

meta_chunks = []
for f in meta_files:
    df = pd.read_csv(f)
    meta_chunks.append(df[df['object_id'].isin(all_valid_ids['object_id'])])
combined_meta = pd.concat(meta_chunks, ignore_index=True)

# 3. Extract Dynamic Lightcurves from all 12 files
print("Scanning Lightcurve files (This will take a few minutes)...")
lc_files = glob.glob(os.path.join(DATA_DIR, '*lightcurves*.csv')) + \
           glob.glob(os.path.join(DATA_DIR, '*batch*.csv'))

lc_chunks = []
for f in lc_files:
    print(f"  Filtering {os.path.basename(f)}...")
    chunk = preprocessing.filter_and_load_chunks(f, all_valid_ids)
    lc_chunks.append(chunk)
combined_lc = pd.concat(lc_chunks, ignore_index=True)

# 4. Feature Engineering & Scaling
print("\nEngineering features and scaling...")
combined_lc = preprocessing.engineer_temporal_features(combined_lc)
combined_lc, combined_meta, flux_scaler = preprocessing.scale_features(combined_lc, combined_meta)

# Merge static and dynamic
full_processed_df = combined_lc.merge(combined_meta, on='object_id', how='left')

# 5. Split back into Train and Val
train_df = full_processed_df[full_processed_df['object_id'].isin(train_ids_df['object_id'])]
val_df = full_processed_df[full_processed_df['object_id'].isin(val_ids_df['object_id'])]

# 6. Build Datasets
print("\nBuilding PyTorch Datasets...")
train_dataset = PLAsTiCCTFTDataset(train_df, max_seq_len=350)
val_dataset = PLAsTiCCTFTDataset(val_df, max_seq_len=350)

# 7. Build the Weighted Random Sampler
print("Calculating class weights for the Sampler...")

# Create a fast dictionary mapping object_id -> true_target
target_dict = train_df.groupby('object_id')['true_target'].first().to_dict()

# Count occurrences of each mapped target to calculate inverse frequencies
target_counts = pd.Series(list(target_dict.values())).value_counts().to_dict()
mapped_counts = {train_dataset.target_map[k]: v for k, v in target_counts.items()}
class_weights = {k: 1.0 / v for k, v in mapped_counts.items()}

# Generate the sample weights using lightning-fast dictionary lookups
sample_weights = [class_weights[train_dataset.target_map[target_dict[obj]]] 
                  for obj in train_dataset.object_ids]

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# Create DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("--- PHASE 3 COMPLETE ---")

--- STARTING PHASE 3: DATA PIPELINE ---
Scanning Metadata files...
Scanning Lightcurve files (This will take a few minutes)...
  Filtering plasticc_train_lightcurves.csv...
  Filtering plasticc_test_set_batch10.csv...
  Filtering plasticc_test_set_batch4.csv...
  Filtering plasticc_test_set_batch7.csv...
  Filtering plasticc_test_set_batch5.csv...
  Filtering plasticc_test_set_batch9.csv...
  Filtering plasticc_test_set_batch1.csv...
  Filtering plasticc_test_set_batch2.csv...
  Filtering plasticc_test_set_batch8.csv...
  Filtering plasticc_test_set_batch3.csv...
  Filtering plasticc_test_set_batch6.csv...
  Filtering plasticc_test_set_batch11.csv...

Engineering features and scaling...

Building PyTorch Datasets...
Calculating class weights for the Sampler...
--- PHASE 3 COMPLETE ---


## Phase 3.5: Pipeline Sanity Checks
Before initializing the neural network, we must mathematically verify the outputs of Phase 3. This block ensures dataset sizes align with our splits, validates the 3D tensor shapes for the GPU, and pulls a single batch to prove the `WeightedRandomSampler` is correctly pulling rare anomalies instead of just majority classes.

In [6]:
print("--- PHASE 3.5: PIPELINE SANITY CHECKS ---")

# 1. Dataset Size Verification
print("1. Dataset Size Verification:")
print(f"   Train Dataset Objects: {len(train_dataset)} (Expected: {len(train_ids_df)})")
print(f"   Val Dataset Objects:   {len(val_dataset)} (Expected: {len(val_ids_df)})")

# If these fail, some objects from your splits weren't found in the raw Kaggle files
assert len(train_dataset) == len(train_ids_df), "CRITICAL: Mismatch in training set size!"
assert len(val_dataset) == len(val_ids_df), "CRITICAL: Mismatch in validation set size!"
print("   -> Sizes match perfectly.")

# 2. Tensor Shape Verification
print("\n2. Tensor Shape Verification:")
sample_batch = next(iter(train_loader))

print(f"   - static:   {sample_batch['static'].shape} -> Expected: [{BATCH_SIZE}, 3]")
print(f"   - dyn_cont: {sample_batch['dyn_cont'].shape} -> Expected: [{BATCH_SIZE}, 350, 4]")
print(f"   - dyn_cat:  {sample_batch['dyn_cat'].shape} -> Expected: [{BATCH_SIZE}, 350]")
print(f"   - mask:     {sample_batch['mask'].shape} -> Expected: [{BATCH_SIZE}, 350]")
print(f"   - target:   {sample_batch['target'].shape} -> Expected: [{BATCH_SIZE}]")

# 3. Weighted Sampler Verification
print("\n3. Weighted Sampler Distribution (Checking 1 Batch):")
batch_targets = sample_batch['target'].numpy()
unique_classes, counts = np.unique(batch_targets, return_counts=True)

# Map internal PyTorch indices (0, 1, 2) back to physical IDs (42, 64, 90)
ordered_original_classes = sorted(train_dataset.target_map.keys())

print("   Physical Class ID : Occurrences in this batch")
for cls_idx, count in zip(unique_classes, counts):
    physical_id = ordered_original_classes[cls_idx]
    print(f"   - Class {physical_id:02d}: {count}")

print("\n--- SANITY CHECKS PASSED ---")

--- PHASE 3.5: PIPELINE SANITY CHECKS ---
1. Dataset Size Verification:
   Train Dataset Objects: 100000 (Expected: 100000)
   Val Dataset Objects:   15000 (Expected: 15000)
   -> Sizes match perfectly.

2. Tensor Shape Verification:
   - static:   torch.Size([64, 3]) -> Expected: [64, 3]
   - dyn_cont: torch.Size([64, 350, 4]) -> Expected: [64, 350, 4]
   - dyn_cat:  torch.Size([64, 350]) -> Expected: [64, 350]
   - mask:     torch.Size([64, 350]) -> Expected: [64, 350]
   - target:   torch.Size([64]) -> Expected: [64]

3. Weighted Sampler Distribution (Checking 1 Batch):
   Physical Class ID : Occurrences in this batch
   - Class 06: 3
   - Class 15: 3
   - Class 16: 8
   - Class 42: 2
   - Class 52: 1
   - Class 53: 1
   - Class 62: 2
   - Class 64: 2
   - Class 67: 3
   - Class 88: 5
   - Class 90: 5
   - Class 92: 8
   - Class 95: 1
   - Class 991: 8
   - Class 992: 7
   - Class 993: 2
   - Class 994: 3

--- SANITY CHECKS PASSED ---


## Phase 4 & 5: Optuna Bayesian Sweep & Training Loop

To find the optimal model capacity and learning dynamics without blindly wasting compute, we have combined the model initialization (Phase 4) and the training loop (Phase 5) into an **Optuna Bayesian Optimization Study**.

* **Intelligent Search:** Optuna uses a Tree-structured Parzen Estimator to actively learn which hyperparameter combinations (learning rate, dropout, network depth) yield the best validation Log-Loss.
* **Pruning:** Unpromising trials are killed early to save GPU time.
* **Aggressive Memory Management:** To prevent Out-Of-Memory (OOM) crashes on Kaggle's strict 30GB RAM limit, we execute garbage collection (`gc.collect()`) and clear the GPU cache (`torch.cuda.empty_cache()`) at the end of every single epoch.

In [7]:
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
import optuna
import gc
import torch

print("--- STARTING PHASE 4 & 5: OPTUNA BAYESIAN SWEEP (BF16) ---")

global_best_loss = float('inf')

def objective(trial):
    global global_best_loss
    
    # 1. Dynamically Sample Hyperparameters
    d_model = trial.suggest_categorical('d_model', [64, 128, 256])
    nhead = trial.suggest_categorical('nhead', [4, 8]) 
    num_layers = trial.suggest_int('num_layers', 2, 3)
    dropout = trial.suggest_float('dropout', 0.2, 0.4)
    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    # 2. Initialize Model
    model = TFTClassifier(
        num_classes=train_dataset.num_classes, 
        d_model=d_model, 
        nhead=nhead, 
        num_layers=num_layers, 
        dropout=dropout
    ).to(device)

    # 3. Custom Weighted Loss
    ordered_original_classes = sorted(train_dataset.target_map.keys())
    loss_weights = [2.0 if cls in [64, 99] else 1.0 for cls in ordered_original_classes]
    loss_weight_tensor = torch.tensor(loss_weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=loss_weight_tensor)

    # 4. Optimizer, Scheduler & Scaler
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    scaler = GradScaler('cuda')

    EPOCHS = 15 
    
    for epoch in range(EPOCHS):
        # --- TRAINING ---
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            
            static, dyn_cont, dyn_cat, mask, targets = (
                batch['static'].to(device), batch['dyn_cont'].to(device),
                batch['dyn_cat'].to(device), batch['mask'].to(device), batch['target'].to(device)
            )
            
            # --- BF16 FIX IMPLEMENTED HERE ---
            with autocast('cuda', dtype=torch.bfloat16):
                logits = model(static, dyn_cont, dyn_cat, mask)
                loss = criterion(logits, targets)
            
            if torch.isnan(loss):
                print(f"Trial {trial.number} failed: NaN loss detected.")
                raise optuna.TrialPruned()
                
            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
        scheduler.step()
        
        del static, dyn_cont, dyn_cat, mask, targets, logits, loss
        torch.cuda.empty_cache()

        # --- VALIDATION ---
        model.eval()
        val_preds_proba, val_true_targets = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                static, dyn_cont, dyn_cat, mask, targets = (
                    batch['static'].to(device), batch['dyn_cont'].to(device),
                    batch['dyn_cat'].to(device), batch['mask'].to(device), batch['target'].to(device)
                )
                
                # --- BF16 FIX IMPLEMENTED HERE ---
                with autocast('cuda', dtype=torch.bfloat16):
                    logits = model(static, dyn_cont, dyn_cat, mask)
                    probs = torch.softmax(logits, dim=1)
                    
                val_preds_proba.extend(probs.cpu().numpy())
                val_true_targets.extend(targets.cpu().numpy())
                
        del static, dyn_cont, dyn_cat, mask, targets, logits, probs
        torch.cuda.empty_cache()

        val_preds_proba = np.array(val_preds_proba)
        
        if np.isnan(val_preds_proba).any():
             print(f"Trial {trial.number} failed: NaN predictions detected.")
             raise optuna.TrialPruned()
             
        val_true_targets = np.array(val_true_targets)
        mapped_true_physical = [ordered_original_classes[idx] for idx in val_true_targets]
        
        k_log_loss = metrics.plasticc_log_loss(mapped_true_physical, val_preds_proba, ordered_original_classes)
        
        if k_log_loss < global_best_loss:
            global_best_loss = k_log_loss
            torch.save(model.state_dict(), '/kaggle/working/best_tft_model.pth')
            
        trial.report(k_log_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return k_log_loss

pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=3)
study = optuna.create_study(direction='minimize', pruner=pruner)

print("Beginning Optuna Sweep...")
study.optimize(objective, n_trials=10, gc_after_trial=True)

print("\n--- SWEEP COMPLETE ---")
print("Best Trial Parameters:")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")
print(f"Best Validation Log-Loss: {study.best_value:.4f}")

[I 2026-03-07 17:05:55,279] A new study created in memory with name: no-name-5d8d8e85-7089-4252-8c51-c1cb18339072


--- STARTING PHASE 4 & 5: OPTUNA BAYESIAN SWEEP (BF16) ---
Beginning Optuna Sweep...


[I 2026-03-07 18:25:23,170] Trial 0 finished with value: 0.23401787700254242 and parameters: {'d_model': 256, 'nhead': 4, 'num_layers': 2, 'dropout': 0.3278716702182421, 'lr': 0.0009124513035330606, 'weight_decay': 0.00012149990939861167}. Best is trial 0 with value: 0.23401787700254242.
[I 2026-03-07 19:49:34,384] Trial 1 finished with value: 0.09644490105316221 and parameters: {'d_model': 128, 'nhead': 8, 'num_layers': 3, 'dropout': 0.3746343221848242, 'lr': 0.00014797462533671873, 'weight_decay': 0.00026003755484553347}. Best is trial 1 with value: 0.09644490105316221.
[I 2026-03-07 20:38:13,889] Trial 2 finished with value: 0.10939686756295719 and parameters: {'d_model': 128, 'nhead': 4, 'num_layers': 2, 'dropout': 0.26288347867125395, 'lr': 0.0004561455424463825, 'weight_decay': 0.00012055940678574788}. Best is trial 1 with value: 0.09644490105316221.
[I 2026-03-07 22:02:23,909] Trial 3 finished with value: 0.09680101808634434 and parameters: {'d_model': 128, 'nhead': 8, 'num_laye


--- SWEEP COMPLETE ---
Best Trial Parameters:
    d_model: 128
    nhead: 8
    num_layers: 3
    dropout: 0.3746343221848242
    lr: 0.00014797462533671873
    weight_decay: 0.00026003755484553347
Best Validation Log-Loss: 0.0964


## Phase 6: Test Set Inference & Evaluation 

This phase loads the best saved model weights, processes the `test_ids.csv` data using our established memory-safe pipeline, runs a half-precision inference loop, and prints out our full suite of custom mathematical metrics (Weighted Log-Loss, Macro F1, PR-AUC, and Brier Score) directly in the notebook.

In [8]:
import glob
import torch
import gc
import os
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from torch.amp import autocast
from sklearn.preprocessing import RobustScaler

print("--- STARTING PHASE 6: CHUNKED TEST INFERENCE ---")

# 1. SYSTEM RAM CLEARING
print("Executing Garbage Collection...")
try:
    del train_loader, val_loader, train_dataset, val_dataset
    del combined_lc, combined_meta, full_processed_df, train_df, val_df
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("System RAM and GPU VRAM wiped successfully.")

# 2. Re-Initialize Best Model
best_params = study.best_trial.params
model = TFTClassifier(
    num_classes=len(ordered_original_classes), 
    d_model=best_params['d_model'], 
    nhead=best_params['nhead'], 
    num_layers=best_params['num_layers'], 
    dropout=best_params['dropout']
).to(device)

model.load_state_dict(torch.load('/kaggle/working/best_tft_model.pth'))
model.eval()
print("Successfully loaded optimal TFT architecture.")

# 3. Load Test IDs and Metadata
test_ids_df = pd.read_csv(os.path.join(SPLITS_DIR, 'test_ids.csv'))
print(f"Loaded {len(test_ids_df)} Test IDs.")

meta_chunks = []
for f in meta_files:
    df = pd.read_csv(f)
    meta_chunks.append(df[df['object_id'].isin(test_ids_df['object_id'])])
test_meta = pd.concat(meta_chunks, ignore_index=True)

print("Rebuilding static metadata scaler from training set...")
train_meta_path = os.path.join(DATA_DIR, 'plasticc_train_metadata.csv')
full_train_meta = pd.read_csv(train_meta_path)
static_cols = ['hostgal_photoz', 'hostgal_photoz_err', 'mwebv']

rebuilt_static_scaler = RobustScaler()
rebuilt_static_scaler.fit(full_train_meta[static_cols]) # Fit ONLY on training data
test_meta[static_cols] = rebuilt_static_scaler.transform(test_meta[static_cols]) # Transform test data
del full_train_meta
gc.collect()

test_preds_proba = []
test_true_targets = []

# 4. CHUNKED INFERENCE LOOP (RAM SAFE MICRO-BATCHING)
print("\nCommencing RAM-Safe Micro-Batched Inference...")
for f in lc_files:
    print(f"  Processing {os.path.basename(f)}...")
    
    # Load the raw file (This stays safely around ~6.5GB)
    chunk_lc = preprocessing.filter_and_load_chunks(f, test_ids_df)
    if chunk_lc.empty:
        continue
        
    # --- THE MICRO-BATCHING FIX ---
    unique_ids = chunk_lc['object_id'].unique()
    micro_batch_size = 10000 
    print(f"    Found {len(unique_ids)} objects. Slicing into {max(1, len(unique_ids)//micro_batch_size)} micro-batches...")
    
    for i in range(0, len(unique_ids), micro_batch_size):
        target_ids = unique_ids[i : i + micro_batch_size]
        
        # Isolate the raw lightcurves for just these 10,000 objects
        micro_lc = chunk_lc[chunk_lc['object_id'].isin(target_ids)].copy()
        
        # Engineer & Scale safely inside the loop
        micro_lc = preprocessing.engineer_temporal_features(micro_lc)
        micro_lc[['flux', 'flux_err']] = flux_scaler.transform(micro_lc[['flux', 'flux_err']])
        
        # Merge to metadata
        micro_df = micro_lc.merge(test_meta, on='object_id', how='left')
        
        # Build Dataset & DataLoader
        micro_dataset = PLAsTiCCTFTDataset(micro_df, max_seq_len=350)
        micro_loader = DataLoader(micro_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
        
        # GPU Inference
        with torch.no_grad():
            for batch in micro_loader:
                static, dyn_cont, dyn_cat, mask, targets = (
                    batch['static'].to(device), batch['dyn_cont'].to(device),
                    batch['dyn_cat'].to(device), batch['mask'].to(device), batch['target'].to(device)
                )
                
                # Force BF16 to prevent overflow
                with autocast('cuda', dtype=torch.bfloat16):
                    logits = model(static, dyn_cont, dyn_cat, mask)
                    probs = torch.softmax(logits, dim=1)
                    
                test_preds_proba.extend(probs.cpu().numpy())
                test_true_targets.extend(targets.cpu().numpy())
                
        # Aggressive RAM wipe for the micro-batch
        del micro_lc, micro_df, micro_dataset, micro_loader, target_ids
        gc.collect()
        
    # Clean up the large chunk once all micro-batches are done
    del chunk_lc, unique_ids
    gc.collect()

# 5. Final Metric Calculations
test_preds_proba = np.array(test_preds_proba)
test_true_targets = np.array(test_true_targets)
test_pred_classes = np.argmax(test_preds_proba, axis=1)

mapped_test_true_physical = [ordered_original_classes[idx] for idx in test_true_targets]

print("\n--- FINAL TEST SET METRICS ---")
test_log_loss = metrics.plasticc_log_loss(mapped_test_true_physical, test_preds_proba, ordered_original_classes)
print(f"PLAsTiCC Weighted Log-Loss: {test_log_loss:.4f}")

test_macro_f1 = metrics.macro_f1(test_true_targets, test_pred_classes)
print(f"Macro F1-Score:             {test_macro_f1:.4f}")

test_pr_auc = metrics.macro_pr_auc(test_true_targets, test_preds_proba, list(range(len(ordered_original_classes))))
print(f"Macro PR-AUC:               {test_pr_auc:.4f}")

test_brier = metrics.multiclass_brier_score(test_true_targets, test_preds_proba, list(range(len(ordered_original_classes))))
print(f"Multiclass Brier Score:     {test_brier:.4f}")

print("\n--- PIPELINE COMPLETE ---")

--- STARTING PHASE 6: CHUNKED TEST INFERENCE ---
Executing Garbage Collection...
System RAM and GPU VRAM wiped successfully.
Successfully loaded optimal TFT architecture.
Loaded 3385738 Test IDs.
Rebuilding static metadata scaler from training set...

Commencing RAM-Safe Micro-Batched Inference...
  Processing plasticc_train_lightcurves.csv...
    Found 7580 objects. Slicing into 1 micro-batches...
  Processing plasticc_test_set_batch10.csv...
    Found 334719 objects. Slicing into 33 micro-batches...
  Processing plasticc_test_set_batch4.csv...
    Found 334550 objects. Slicing into 33 micro-batches...
  Processing plasticc_test_set_batch7.csv...
    Found 334747 objects. Slicing into 33 micro-batches...
  Processing plasticc_test_set_batch5.csv...
    Found 334649 objects. Slicing into 33 micro-batches...
  Processing plasticc_test_set_batch9.csv...
    Found 334597 objects. Slicing into 33 micro-batches...
  Processing plasticc_test_set_batch1.csv...
    Found 31862 objects. Slicing